<a href="https://colab.research.google.com/github/TriambigaKrubhakaran/_VITECHAbusiveTextDravidianLangtech2026_/blob/main/Google_gemma_2_9B_Zero_Shot_Code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Zero-Shot Classification using Google Gemma-2-9B BASE MODEL (COLAB FREE TIER)
# Dataset: https://docs.google.com/spreadsheets/d/1XbsUFrSAEP125Paohepu0fdFB8sIeezQKTe3ImBb2o0/
# Model: google/gemma-2-9b (Base model)
# Optimized for Tamil text with 4-bit quantization
# PREDICTION ONLY VERSION - No evaluation metrics

# ============================================================
# INSTALL REQUIRED LIBRARIES
# ============================================================
!pip install transformers torch accelerate bitsandbytes huggingface_hub -q

import pandas as pd
import numpy as np
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from tqdm import tqdm
import torch
import time
from datetime import datetime
import os
import random
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# HUGGING FACE AUTHENTICATION
# ============================================================
from huggingface_hub import login
from getpass import getpass

print("="*80)
print("GOOGLE GEMMA-2-9B ZERO-SHOT TAMIL CLASSIFICATION")
print("PREDICTION ONLY MODE")
print("="*80)
print(f"Started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("="*80 + "\n")

print("🔐 HuggingFace Authentication Required")
print("-"*80)
print("1. Get your token from: https://huggingface.co/settings/tokens")
print("2. Accept Gemma-2 license: https://huggingface.co/google/gemma-2-9b")
print("-"*80)

hf_token = getpass("Enter HuggingFace Token: ")

if not hf_token or hf_token.strip() == "":
    raise ValueError("❌ Token cannot be empty")

login(token=hf_token)
print("✓ Authenticated successfully\n")

# ============================================================
# LOAD DATASET
# ============================================================
print("="*80)
print("Step 1: Loading Dataset")
print("="*80)

sheet_id = '1GnV9u0kVqLH_sfBRY4AI-ZcxZzrmybd907mpY6DHHf0'
gid = '2031377137'
url = f'https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv&gid={gid}'

try:
    df = pd.read_csv(url)
    print(f"✓ Loaded {len(df)} rows")
    print(f"Columns: {df.columns.tolist()}\n")
except Exception as e:
    raise Exception(f"❌ Error loading dataset: {e}")

# Auto-detect text column
text_col = next((c for c in ['Text', 'text', 'Comment', 'comment', 'Content']
                 if c in df.columns), df.columns[0])

print(f"Text column: '{text_col}'")
print()

# ============================================================
# LOAD GOOGLE GEMMA-2-9B WITH 4-BIT QUANTIZATION
# ============================================================
print("="*80)
print("Step 2: Loading Google Gemma-2-9B (4-bit quantized)")
print("="*80)

model_name = "google/gemma-2-9b"  # BASE MODEL

# 4-bit quantization configuration (ESSENTIAL for free tier)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,  # Gemma-2 prefers bfloat16
    bnb_4bit_use_double_quant=True,
)

print(f"Model: {model_name}")
print("Loading model... (this may take 2-3 minutes)")
print("Configuration: 4-bit NF4 quantization with bfloat16 for T4 GPU\n")

try:
    tokenizer = AutoTokenizer.from_pretrained(
        model_name,
        token=hf_token,
        trust_remote_code=True
    )

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        token=hf_token,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
        low_cpu_mem_usage=True
    )

    # Gemma-2 specific tokenizer setup
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        tokenizer.pad_token_id = tokenizer.eos_token_id

    model.eval()
    print("✓ Model loaded successfully!\n")

except Exception as e:
    print(f"❌ Error loading model: {e}\n")
    print("Troubleshooting:")
    print("1. Enable GPU: Runtime > Change runtime type > T4 GPU")
    print("2. Accept Gemma-2 license: https://huggingface.co/google/gemma-2-9b")
    print("3. Verify your HF token has access to gated models")
    raise

# ============================================================
# DEVICE INFORMATION
# ============================================================
print("-"*80)
print("Device Configuration:")
print("-"*80)
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"Total GPU Memory: {mem_gb:.1f} GB")
    allocated = torch.cuda.memory_allocated(0) / 1e9
    print(f"Allocated: {allocated:.1f} GB")
    print(f"Model device: {next(model.parameters()).device}")
else:
    print("⚠️  WARNING: Running on CPU - will be extremely slow!")
    print("Enable GPU: Runtime > Change runtime type > T4 GPU")
print("-"*80 + "\n")

# ============================================================
# GENERATION PARAMETERS (OPTIMIZED FOR GEMMA-2 BASE MODEL)
# ============================================================
GENERATION_PARAMS = {
    "max_new_tokens": 15,
    "temperature": 0.4,
    "do_sample": True,
    "top_p": 0.9,
    "top_k": 40,
    "repetition_penalty": 1.15,
    "pad_token_id": tokenizer.pad_token_id,
    "eos_token_id": tokenizer.eos_token_id,
}

print("="*80)
print("Generation Parameters (Optimized for Gemma-2 Base Model)")
print("="*80)
for key, value in GENERATION_PARAMS.items():
    print(f"  {key}: {value}")
print("="*80 + "\n")

# ============================================================
# CLASSIFICATION FUNCTION FOR GEMMA-2 BASE MODEL
# ============================================================
def classify_text_gemma2(text, verbose=False):
    """
    Zero-shot classification using Gemma-2-9B base model.
    Uses completion-style prompts optimized for Tamil text.
    """
    if pd.isna(text) or str(text).strip() == "":
        return "non-abusive"

    text = str(text).strip()[:400]

    # Completion-style prompt for base model
    # Gemma-2 works well with clear, structured prompts
    prompt = f"""Classify the following text as either "abusive" or "non-abusive".

Text: {text}

Classification:"""

    try:
        inputs = tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=450,
            padding=True
        ).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                **GENERATION_PARAMS
            )

        response = tokenizer.decode(
            outputs[0][inputs['input_ids'].shape[1]:],
            skip_special_tokens=True
        ).strip()

        if verbose:
            print(f"Raw response: '{response}'")

        response_lower = response.lower().strip()

        # Remove common response artifacts
        response_lower = response_lower.replace('\n', ' ').strip()

        # Clean common prefixes/phrases
        prefixes = [
            "the text is", "this text is", "it is", "classified as",
            "categorized as", "considered", "labeled as", "answer:",
            "classification:", "**", "*"
        ]
        for prefix in prefixes:
            if response_lower.startswith(prefix):
                response_lower = response_lower.replace(prefix, "", 1).strip()

        if verbose:
            print(f"Cleaned: '{response_lower}'")

        # Parsing logic
        # Stage 1: Exact matches
        if response_lower == "non-abusive" or response_lower == "not abusive":
            return "non-abusive"
        elif response_lower == "abusive":
            return "abusive"

        # Stage 2: Starts with (first 20 chars)
        first_part = response_lower[:20]
        if first_part.startswith("non-abusive") or first_part.startswith("not abusive"):
            return "non-abusive"
        elif first_part.startswith("abusive") and "non" not in first_part:
            return "abusive"

        # Stage 3: Word-level analysis
        words = response_lower.split()[:10]  # First 10 words

        # Check for non-abusive indicators
        non_abusive_indicators = ["non-abusive", "not abusive", "non abusive"]
        for indicator in non_abusive_indicators:
            if indicator in response_lower:
                return "non-abusive"

        # Check for abusive (ensure no negation)
        if "abusive" in response_lower:
            abusive_pos = response_lower.index("abusive")
            before_text = response_lower[:abusive_pos]
            negation_words = ["non", "not", "no", "isn't", "neither"]
            if not any(neg in before_text.split()[-3:] for neg in negation_words):
                return "abusive"

        # Stage 4: Sentiment/tone indicators
        negative_indicators = ["offensive", "harmful", "hate", "violent", "toxic"]
        positive_indicators = ["neutral", "normal", "acceptable", "fine", "ok"]

        if any(word in response_lower for word in negative_indicators):
            return "abusive"
        elif any(word in response_lower for word in positive_indicators):
            return "non-abusive"

        # Stage 5: Unbiased random fallback
        if verbose:
            print(f"⚠️  UNPARSEABLE: '{response}' - using random fallback")

        return random.choice(["abusive", "non-abusive"])

    except Exception as e:
        if verbose:
            print(f"❌ Error: {e}")
        return random.choice(["abusive", "non-abusive"])

# ============================================================
# SPEED TEST
# ============================================================
print("="*80)
print("Step 3: Speed Test (5 samples)")
print("="*80 + "\n")

test_size = min(5, len(df))
df_test = df.head(test_size).copy()

predictions_test = []
start_time = time.time()

print("Sample predictions:\n")
for idx in range(test_size):
    text = df_test.iloc[idx][text_col]

    verbose = (idx < 2)  # Show details for first 2
    pred = classify_text_gemma2(text, verbose=verbose)
    predictions_test.append(pred)

    print(f"{idx+1}. Predicted: {pred}")
    print(f"   Text: {str(text)[:70]}...")
    if verbose:
        print()

test_time = time.time() - start_time
avg_time = test_time / test_size

print("-"*80)
print(f"Speed test completed:")
print(f"  Time: {test_time:.1f}s for {test_size} samples")
print(f"  Average: {avg_time:.2f} seconds/text")
print(f"\nEstimate for full dataset ({len(df)} texts):")
estimated_mins = (avg_time * len(df)) / 60
estimated_hours = estimated_mins / 60
print(f"  ~{estimated_mins:.0f} minutes ({estimated_hours:.1f} hours)")
print("="*80 + "\n")

# ============================================================
# USER CONFIRMATION
# ============================================================
print("⚠️  CHECKPOINT: Processing Mode")
print("-"*80)
print(f"You can process:")
print(f"  1. FULL dataset: {len(df)} texts (~{estimated_hours:.1f} hours)")
print(f"  2. SAMPLE: 500 texts (~{(avg_time * 500) / 60:.0f} minutes)")
print("-"*80)

PROCESS_FULL = True  # Change to False for sample mode
SAMPLE_SIZE = 500

if not PROCESS_FULL:
    df = df.head(SAMPLE_SIZE)
    print(f"\n✓ Processing SAMPLE: {len(df)} texts\n")
else:
    print(f"\n✓ Processing FULL DATASET: {len(df)} texts")
    print(f"   Estimated time: ~{estimated_hours:.1f} hours\n")

# ============================================================
# PROCESS DATASET
# ============================================================
print("="*80)
print(f"Step 4: Zero-Shot Classification")
print(f"Processing {len(df)} texts...")
print("="*80 + "\n")

predictions = []
start_time = time.time()
batch_size = 50

for idx, text in enumerate(tqdm(df[text_col], desc="Classifying")):
    pred = classify_text_gemma2(text)
    predictions.append(pred)

    # Batch progress updates
    if (idx + 1) % batch_size == 0:
        elapsed = time.time() - start_time
        avg = elapsed / (idx + 1)
        remaining = avg * (len(df) - idx - 1)

        print(f"\n[{idx+1}/{len(df)}] Avg: {avg:.2f}s/text | ETA: {remaining/60:.0f}m")

    # Clear CUDA cache periodically
    if (idx + 1) % 100 == 0 and torch.cuda.is_available():
        torch.cuda.empty_cache()

total_time = time.time() - start_time
print(f"\n✓ Classification completed!")
print(f"  Total time: {total_time/60:.1f} minutes ({total_time/3600:.2f} hours)")
print(f"  Average: {total_time/len(df):.2f} seconds/text\n")

# ============================================================
# PREPARE OUTPUT
# ============================================================
df['predicted_label'] = predictions

# ============================================================
# SAVE RESULTS
# ============================================================
print("="*80)
print("Step 5: Saving Results")
print("="*80)

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

# Mount Google Drive
try:
    from google.colab import drive
    if not os.path.exists('/content/drive'):
        print("\nMounting Google Drive...")
        drive.mount('/content/drive')

    output_dir = '/content/drive/MyDrive/Gemma2_Tamil_Results/'
    os.makedirs(output_dir, exist_ok=True)
    print(f"✓ Saving to: {output_dir}")
    drive_mounted = True
except:
    output_dir = '/content/Gemma2_Results/'
    os.makedirs(output_dir, exist_ok=True)
    print(f"✓ Saving locally to: {output_dir}")
    drive_mounted = False

# Save prediction file
output_file = f'{output_dir}predictions_gemma2_9b_{timestamp}.csv'
df[[text_col, 'predicted_label']].to_csv(
    output_file, index=False, encoding='utf-8-sig'
)
print(f"\n✓ Saved: predictions_gemma2_9b_{timestamp}.csv")

# Save summary
summary_file = f'{output_dir}summary_{timestamp}.txt'
with open(summary_file, 'w', encoding='utf-8') as f:
    f.write("="*80 + "\n")
    f.write("GOOGLE GEMMA-2-9B - ZERO-SHOT TAMIL CLASSIFICATION\n")
    f.write("PREDICTION ONLY MODE\n")
    f.write("="*80 + "\n\n")
    f.write(f"Model: {model_name}\n")
    f.write(f"Method: Zero-Shot (Base model)\n")
    f.write(f"Dataset: {len(df)} texts (Tamil)\n")
    f.write(f"Processing time: {total_time/60:.1f} min ({total_time/3600:.2f} hrs)\n")
    f.write(f"Average speed: {total_time/len(df):.2f} sec/text\n\n")
    f.write("="*80 + "\n")
    f.write("PREDICTION DISTRIBUTION\n")
    f.write("="*80 + "\n")
    f.write(str(df['predicted_label'].value_counts()))
    f.write("\n\n" + "="*80 + "\n")
    f.write("CONFIGURATION\n")
    f.write("="*80 + "\n")
    f.write("Quantization: 4-bit NF4 with bfloat16 (for Colab T4 GPU)\n")
    f.write("Prompt style: Completion-based (optimized for Tamil)\n")
    for k, v in GENERATION_PARAMS.items():
        f.write(f"{k}: {v}\n")

print(f"✓ Saved: summary_{timestamp}.txt")

print("\n" + "="*80)
print("✅ PIPELINE COMPLETE!")
print("="*80)
print(f"Model: Google Gemma-2-9B")
print(f"Processed: {len(df)} texts")
print(f"Time: {total_time/60:.1f} min ({total_time/len(df):.2f}s per text)")
print(f"\nPrediction distribution:")
print(df['predicted_label'].value_counts())
print(f"\nResults saved to: {output_dir}")
print("="*80)